# Credit Cluster Scoring Tool

Use this notebook to score a private, simulated, or competitor company against the saved credit clustering model.

Expected workflow:

1. Put the saved artifact at `saved_models/credit_cluster_model_v1.joblib`.
2. Enter company financials manually, or load a CSV/Excel file.
3. Run the scoring cells.
4. Review cluster assignment, near-default affinity, feature coverage, and warning flags.

Important: the output is **not a formal credit rating** and not a calibrated probability of default. It is a public-company financial profile match based on distance to the learned KMeans credit clusters.

In [18]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import sys

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Load saved fitted model artifact

The training notebook should have saved this file after fitting the KMeans model.

In [19]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

MODEL_PATH = PROJECT_ROOT / "saved_models/nonfinancial_credit_scorecard_kmeans_k5_v3.joblib"

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model artifact not found at {MODEL_PATH}. "
        "Copy nonfinancial_credit_kmeans_k5.joblib into saved_models/ or update MODEL_PATH."
    )

artifact = joblib.load(MODEL_PATH)

print("Loaded artifact:", artifact.get("model_name"))
print("Segment:", artifact.get("segment"))
print("Training rows:", artifact.get("training_rows"))
print("Artifact keys:", list(artifact.keys()))

pipe = artifact["pipeline"]
feature_cols = artifact["feature_cols"]
cluster_labels = artifact["cluster_labels"]
risk_rank = artifact["risk_rank"]
winsor_caps = artifact.get("winsor_caps")

print("Features used by model:", feature_cols)
print("Cluster labels:", cluster_labels)

Loaded artifact: nonfinancial_credit_scorecard_kmeans_k5_v3
Segment: Non-financial
Training rows: 21265
Artifact keys: ['model_name', 'version', 'segment', 'pipeline', 'feature_cols', 'cluster_labels', 'risk_rank', 'scorecard_thresholds', 'scorecard_domain_weights', 'min_non_null_features', 'scorecard_cluster_features', 'scorecard_component_features', 'winsor_caps', 'training_rows', 'cluster_profile_ranked', 'metrics_df', 'notes']
Features used by model: ['structural_distress_risk', 'earnings_risk', 'operating_cashflow_risk', 'liquidity_risk', 'leverage_risk', 'debt_service_risk']
Cluster labels: {4: '1 - Low risk / investment-grade-like', 1: '2 - Moderate risk / lower-investment-grade-like', 0: '3 - Elevated risk / leveraged', 2: '4 - High risk / speculative', 3: '5 - Distressed / near-default proxy'}


## 2. Define feature engineering and scoring helpers

These functions convert raw financial statement inputs into the same ratios used by the clustering model.

In [29]:
def _ensure_columns(df, columns, default=np.nan):
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = default
    return out


def safe_divide(numerator, denominator, min_abs_denominator=None):
    numerator = pd.Series(numerator).astype(float)
    denominator = pd.Series(denominator).astype(float)

    out = numerator / denominator

    if min_abs_denominator is not None:
        out = out.where(denominator.abs() >= min_abs_denominator)

    return out.replace([np.inf, -np.inf], np.nan)


def engineer_private_company_features(input_df, winsor_caps=None, fx_to_model_currency=1.0):
    """
    Convert raw private-company financials into model-ready features.

    This function creates:
    1. Raw financial ratios used for diagnostics/interpretation
    2. Engineered risk features required by the trained clustering model:
       - structural_distress_risk
       - earnings_risk
       - operating_cashflow_risk
       - liquidity_risk
       - leverage_risk
       - debt_service_risk

    fx_to_model_currency:
        Multiplier used to convert absolute monetary values into the model currency.
        If the model was trained on USD and input is EUR, use EURUSD rate.
        For ratio-only sensitivity, this mostly affects log_assets.
    """
    out = input_df.copy()

    required_or_optional = [
        "assets", "liabilities", "equity", "cash", "net_income", "cfo", "revenue",
        "long_term_debt", "short_term_debt", "assets_current", "current_assets",
        "liabilities_current", "current_liabilities", "receivables", "inventory",
        "capex", "operating_income", "gross_profit", "interest_expense",
    ]
    out = _ensure_columns(out, required_or_optional)

    # Accept both EDGAR-style and business-friendly names.
    if out["assets_current"].isna().all() and "current_assets" in out.columns:
        out["assets_current"] = out["current_assets"]
    if out["liabilities_current"].isna().all() and "current_liabilities" in out.columns:
        out["liabilities_current"] = out["current_liabilities"]

    monetary_cols = [
        "assets", "liabilities", "equity", "cash", "net_income", "cfo", "revenue",
        "long_term_debt", "short_term_debt", "assets_current", "current_assets",
        "liabilities_current", "current_liabilities", "receivables", "inventory",
        "capex", "operating_income", "gross_profit", "interest_expense",
    ]

    for col in monetary_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # Convert only absolute numbers; ratios are calculated afterward.
    for col in monetary_cols:
        out[col] = out[col] * fx_to_model_currency

    out["long_term_debt"] = out["long_term_debt"].fillna(0)
    out["short_term_debt"] = out["short_term_debt"].fillna(0)
    out["debt"] = out["long_term_debt"] + out["short_term_debt"]

    out["capex"] = out["capex"].fillna(0)
    out["free_cash_flow"] = out["cfo"] - out["capex"]

    out["log_assets"] = np.log1p(out["assets"].clip(lower=0))

    out["liabilities_to_assets"] = safe_divide(out["liabilities"], out["assets"], 10_000_000)
    out["debt_to_assets"] = safe_divide(out["debt"], out["assets"], 10_000_000)
    out["equity_to_assets"] = safe_divide(out["equity"], out["assets"], 10_000_000)
    out["cash_to_assets"] = safe_divide(out["cash"], out["assets"], 10_000_000)
    out["net_income_to_assets"] = safe_divide(out["net_income"], out["assets"], 10_000_000)
    out["cfo_to_assets"] = safe_divide(out["cfo"], out["assets"], 10_000_000)
    out["revenue_to_assets"] = safe_divide(out["revenue"], out["assets"], 10_000_000)

    out["current_ratio"] = safe_divide(out["assets_current"], out["liabilities_current"], 1_000_000)
    out["quick_ratio"] = safe_divide(
        out["cash"].fillna(0) + out["receivables"].fillna(0),
        out["liabilities_current"],
        1_000_000,
    )

    # Diagnostic-only ratios. These are not necessarily used directly by the model,
    # but they make the score easier to interpret.
    out["operating_margin"] = safe_divide(out["operating_income"], out["revenue"], 10_000_000)
    out["gross_margin"] = safe_divide(out["gross_profit"], out["revenue"], 10_000_000)
    out["interest_coverage"] = safe_divide(out["operating_income"], out["interest_expense"], 1_000_000)
    out["fcf_to_debt"] = safe_divide(out["free_cash_flow"], out["debt"], 1_000_000)

    # Apply the same winsorization/capping used by the training pipeline.
    # This should happen before translating ratios into risk scores.
    if winsor_caps is not None:
        for col, bounds in winsor_caps.items():
            if col in out.columns and bounds is not None:
                lo, hi = bounds
                out[col] = out[col].clip(lower=lo, upper=hi)

    # -------------------------------------------------------------------------
    # Engineered risk features required by the trained clustering model
    # -------------------------------------------------------------------------

    # Balance-sheet leverage risk
    out["liabilities_risk"] = (
        (out["liabilities_to_assets"] - 0.50) / (1.00 - 0.50)
    ).clip(0, 1)

    out["debt_load_risk"] = (
        (out["debt_to_assets"] - 0.25) / (0.75 - 0.25)
    ).clip(0, 1)

    out["equity_buffer_risk"] = (
        (0.40 - out["equity_to_assets"]) / (0.40 - 0.00)
    ).clip(0, 1)

    out["leverage_risk"] = out[
        ["liabilities_risk", "debt_load_risk", "equity_buffer_risk"]
    ].mean(axis=1)

    # Liquidity risk
    out["cash_buffer_risk"] = (
        (0.10 - out["cash_to_assets"]) / 0.10
    ).clip(0, 1)

    out["current_liquidity_risk"] = (
        (1.50 - out["current_ratio"]) / (1.50 - 0.75)
    ).clip(0, 1)

    out["quick_liquidity_risk"] = (
        (1.00 - out["quick_ratio"]) / (1.00 - 0.50)
    ).clip(0, 1)

    out["liquidity_risk"] = out[
        ["cash_buffer_risk", "current_liquidity_risk", "quick_liquidity_risk"]
    ].mean(axis=1)

    # Earnings/profitability risk
    out["profitability_risk"] = (
        (0.03 - out["net_income_to_assets"]) / (0.03 - (-0.10))
    ).clip(0, 1)

    out["earnings_risk"] = out["profitability_risk"]

    # Operating cash-flow risk
    out["cashflow_risk"] = (
        (0.03 - out["cfo_to_assets"]) / (0.03 - (-0.10))
    ).clip(0, 1)

    out["operating_cashflow_risk"] = out["cashflow_risk"]

    # Debt-service risk
    out["coverage_risk"] = (
        (3.0 - out["interest_coverage"]) / (3.0 - 1.0)
    ).clip(0, 1)

    out["fcf_risk"] = (
        (0.05 - out["fcf_to_debt"]) / (0.05 - (-0.25))
    ).clip(0, 1)

    out["debt_service_risk"] = out[
        ["coverage_risk", "fcf_risk"]
    ].mean(axis=1)

    # Structural distress risk
    out["negative_equity_flag"] = (
        out["equity_to_assets"] < 0
    ).astype(float)

    out["liabilities_exceed_assets_flag"] = (
        out["liabilities_to_assets"] > 1
    ).astype(float)

    out["structural_distress_risk"] = out[
        ["negative_equity_flag", "liabilities_exceed_assets_flag"]
    ].max(axis=1)

    return out

def soft_cluster_scores(distances, temperature=1.0):
    distances = np.asarray(distances, dtype=float)
    similarities = np.exp(-distances / temperature)
    return similarities / similarities.sum(axis=1, keepdims=True)


def make_warning_flags(row):
    flags = []

    if pd.notna(row.get("assets")) and row.get("assets") <= 0:
        flags.append("invalid_assets")

    if pd.notna(row.get("assets")) and row.get("assets") < 10_000_000:
        flags.append("assets_below_model_threshold")

    if pd.notna(row.get("liabilities_to_assets")) and row.get("liabilities_to_assets") > 1:
        flags.append("liabilities_exceed_assets")

    if pd.notna(row.get("equity_to_assets")) and row.get("equity_to_assets") < 0:
        flags.append("negative_equity")

    if pd.notna(row.get("debt_to_assets")) and row.get("debt_to_assets") > 0.75:
        flags.append("high_debt_to_assets")

    if pd.notna(row.get("current_ratio")) and row.get("current_ratio") < 1:
        flags.append("current_ratio_below_1")

    if pd.notna(row.get("quick_ratio")) and row.get("quick_ratio") < 0.5:
        flags.append("quick_ratio_below_0_5")

    if pd.notna(row.get("interest_coverage")) and row.get("interest_coverage") < 1:
        flags.append("interest_coverage_below_1")

    if pd.notna(row.get("cfo_to_assets")) and row.get("cfo_to_assets") < 0:
        flags.append("negative_cfo_to_assets")

    return ", ".join(flags) if flags else "none"

def score_companies(input_df, artifact, segment="Non-financial", temperature=1.0, fx_to_model_currency=1.0):
    scored = engineer_private_company_features(
        input_df,
        winsor_caps=artifact.get("winsor_caps"),
        fx_to_model_currency=fx_to_model_currency,
    )

    pipe = artifact["pipeline"]
    feature_cols = artifact["feature_cols"]
    cluster_labels = artifact.get("cluster_labels", {})
    risk_rank = artifact.get("risk_rank", {})

    X_new = scored[feature_cols]

    assigned = pipe.predict(X_new)

    X_prepared = pipe[:-1].transform(X_new)
    distances = pipe.named_steps["cluster"].transform(X_prepared)
    affinities = soft_cluster_scores(distances, temperature=temperature)

    scored["assigned_cluster"] = assigned
    scored["distance_to_assigned_cluster"] = distances[
        np.arange(len(assigned)),
        assigned
        ]
    scored["cluster_label"] = scored["assigned_cluster"].map(cluster_labels)
    scored["risk_rank"] = scored["assigned_cluster"].map(risk_rank)
    scored["cluster_affinity"] = affinities.max(axis=1)

    # Cluster 4 is our near-default / distressed capital structure proxy.
    scored["near_default_affinity"] = affinities[:, 4] if affinities.shape[1] > 4 else np.nan

    for i in range(affinities.shape[1]):
        scored[f"cluster_{i}_affinity"] = affinities[:, i]
        scored[f"cluster_{i}_distance"] = distances[:, i]

    scored["feature_coverage_pct"] = scored[feature_cols].notna().mean(axis=1)

    scored["warning_flags"] = scored.apply(make_warning_flags, axis=1)

    return scored

def _get_cluster_risk_map(artifact):
    """
    Return {cluster_id: risk_rank}.
    Uses artifact["risk_rank"] first, then falls back to artifact["cluster_profile_ranked"].
    """

    if "risk_rank" in artifact and artifact["risk_rank"] is not None:
        risk_rank = artifact["risk_rank"]

        if isinstance(risk_rank, dict):
            return {int(k): int(v) for k, v in risk_rank.items()}

        if isinstance(risk_rank, pd.Series):
            return {int(k): int(v) for k, v in risk_rank.to_dict().items()}

    if "cluster_profile_ranked" in artifact:
        profile = artifact["cluster_profile_ranked"]

        if {"cluster", "risk_rank"}.issubset(profile.columns):
            return {
                int(row["cluster"]): int(row["risk_rank"])
                for _, row in profile[["cluster", "risk_rank"]].dropna().iterrows()
            }

    raise KeyError(
        "Could not infer cluster risk ranks. Expected artifact['risk_rank'] "
        "or artifact['cluster_profile_ranked'] with columns ['cluster', 'risk_rank']."
    )


def _get_cluster_label_map(artifact):
    """
    Return {cluster_id: cluster_label}, if available.
    """

    labels = artifact.get("cluster_labels")

    if labels is None:
        return {}

    if isinstance(labels, dict):
        return {int(k): v for k, v in labels.items()}

    return {}


def _distance_matrix_to_clusters(scored_df, artifact):
    """
    Compute distance from each scored row to each KMeans cluster center.

    Assumes:
    - artifact["pipeline"] is the fitted sklearn Pipeline used for KMeans.
    - artifact["feature_cols"] are the model input features.

    For a sklearn Pipeline ending in KMeans, pipeline.transform(X) returns
    distances to all cluster centers.
    """

    if "pipeline" not in artifact:
        raise KeyError("artifact['pipeline'] not found.")

    if "feature_cols" not in artifact:
        raise KeyError("artifact['feature_cols'] not found.")

    pipeline = artifact["pipeline"]
    feature_cols = artifact["feature_cols"]

    missing_features = [c for c in feature_cols if c not in scored_df.columns]

    if missing_features:
        raise KeyError(
            "The scored dataframe is missing model feature columns required "
            f"for distance calculation: {missing_features}"
        )

    X = scored_df[feature_cols].copy()

    distances = pipeline.transform(X)

    distance_df = pd.DataFrame(
        distances,
        index=scored_df.index,
        columns=[f"distance_to_cluster_{i}" for i in range(distances.shape[1])],
    )

    return distance_df


def add_adjacent_bucket_distances_and_outlook(
    scored_df,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
):
    """
    Adds adjacent bucket distance diagnostics and a cluster-position outlook.

    Added columns:
    - upper_bucket_cluster
    - upper_bucket_label
    - distance_to_upper_bucket
    - lower_bucket_cluster
    - lower_bucket_label
    - distance_to_lower_bucket
    - upper_distance_ratio_to_assigned
    - lower_distance_ratio_to_assigned
    - outlook_flag
    - outlook_reason

    Interpretation:
    - Positive:
        Company is closer to the stronger adjacent bucket and close enough
        to make the upgrade signal meaningful.

    - Neutral:
        Company remains materially closer to its assigned bucket, or the
        difference between adjacent buckets is not strong enough.

    - Negative:
        Company is closer to the weaker adjacent bucket and close enough
        to make the downgrade signal meaningful.

    Important:
    This is a relative cluster-position flag, not a time-series forecast.
    """

    scored_df = scored_df.copy()

    required_cols = [
        "assigned_cluster",
        "risk_rank",
        "distance_to_assigned_cluster",
    ]

    missing_required = [c for c in required_cols if c not in scored_df.columns]

    if missing_required:
        raise KeyError(
            "scored_df is missing required columns: "
            f"{missing_required}"
        )

    risk_map = _get_cluster_risk_map(artifact)
    label_map = _get_cluster_label_map(artifact)
    distance_df = _distance_matrix_to_clusters(scored_df, artifact)

    rank_to_cluster = {
        int(rank): int(cluster)
        for cluster, rank in risk_map.items()
    }

    upper_clusters = []
    lower_clusters = []
    upper_labels = []
    lower_labels = []
    upper_distances = []
    lower_distances = []
    upper_distance_ratios = []
    lower_distance_ratios = []
    outlook_flags = []
    outlook_reasons = []

    for idx, row in scored_df.iterrows():
        assigned_rank = int(row["risk_rank"])
        assigned_distance = pd.to_numeric(
            row["distance_to_assigned_cluster"],
            errors="coerce",
        )

        upper_rank = assigned_rank - 1
        lower_rank = assigned_rank + 1

        upper_cluster = rank_to_cluster.get(upper_rank, np.nan)
        lower_cluster = rank_to_cluster.get(lower_rank, np.nan)

        if pd.notna(upper_cluster):
            upper_cluster = int(upper_cluster)
            upper_distance = distance_df.loc[idx, f"distance_to_cluster_{upper_cluster}"]
            upper_label = label_map.get(upper_cluster, None)
        else:
            upper_distance = np.nan
            upper_label = None

        if pd.notna(lower_cluster):
            lower_cluster = int(lower_cluster)
            lower_distance = distance_df.loc[idx, f"distance_to_cluster_{lower_cluster}"]
            lower_label = label_map.get(lower_cluster, None)
        else:
            lower_distance = np.nan
            lower_label = None

        if pd.notna(assigned_distance) and assigned_distance > 0:
            upper_ratio = upper_distance / assigned_distance if pd.notna(upper_distance) else np.nan
            lower_ratio = lower_distance / assigned_distance if pd.notna(lower_distance) else np.nan
        else:
            upper_ratio = np.nan
            lower_ratio = np.nan

        # -----------------------------------------------------------------
        # Outlook logic.
        #
        # Two-step approach:
        # 1. Direction:
        #    Is the company closer to the stronger or weaker adjacent bucket?
        #
        # 2. Boundary:
        #    Is the adjacent bucket close enough relative to the assigned
        #    bucket to make the signal meaningful?
        #
        # This avoids overstating "Positive" or "Negative" when both adjacent
        # buckets are much farther away than the assigned bucket.
        # -----------------------------------------------------------------
        if pd.isna(assigned_distance):
            outlook = "Neutral"
            reason = (
                "Assigned-cluster distance is missing; outlook cannot be "
                "reliably assessed."
            )

        elif pd.isna(upper_distance) and pd.isna(lower_distance):
            outlook = "Neutral"
            reason = "No adjacent risk buckets available."

        elif pd.isna(upper_distance):
            near_lower = lower_distance <= assigned_distance * downgrade_boundary_multiplier

            if near_lower:
                outlook = "Negative"
                reason = (
                    "Company is near the weaker adjacent bucket and has no "
                    "stronger adjacent bucket available."
                )
            else:
                outlook = "Neutral"
                reason = (
                    "Only the weaker adjacent bucket is available, but the "
                    "company remains materially closer to its assigned bucket."
                )

        elif pd.isna(lower_distance):
            near_upper = upper_distance <= assigned_distance * upgrade_boundary_multiplier

            if near_upper:
                outlook = "Positive"
                reason = (
                    "Company is near the stronger adjacent bucket and has no "
                    "weaker adjacent bucket available."
                )
            else:
                outlook = "Neutral"
                reason = (
                    "Only the stronger adjacent bucket is available, but the "
                    "company remains materially closer to its assigned bucket."
                )

        else:
            diff = lower_distance - upper_distance
            threshold = assigned_distance * neutral_band

            near_upper = upper_distance <= assigned_distance * upgrade_boundary_multiplier
            near_lower = lower_distance <= assigned_distance * downgrade_boundary_multiplier

            if diff > threshold and near_upper:
                outlook = "Positive"
                reason = (
                    "Company is closer to the stronger adjacent bucket and "
                    "close enough to that bucket to indicate positive "
                    "cluster-position outlook."
                )

            elif diff < -threshold and near_lower:
                outlook = "Negative"
                reason = (
                    "Company is closer to the weaker adjacent bucket and "
                    "close enough to that bucket to indicate negative "
                    "cluster-position outlook."
                )

            else:
                outlook = "Neutral"
                reason = (
                    "Company remains materially closer to its assigned bucket "
                    "than to adjacent buckets; no strong upgrade or downgrade "
                    "signal."
                )

        upper_clusters.append(upper_cluster)
        lower_clusters.append(lower_cluster)
        upper_labels.append(upper_label)
        lower_labels.append(lower_label)
        upper_distances.append(upper_distance)
        lower_distances.append(lower_distance)
        upper_distance_ratios.append(upper_ratio)
        lower_distance_ratios.append(lower_ratio)
        outlook_flags.append(outlook)
        outlook_reasons.append(reason)

    scored_df["upper_bucket_cluster"] = upper_clusters
    scored_df["upper_bucket_label"] = upper_labels
    scored_df["distance_to_upper_bucket"] = upper_distances

    scored_df["lower_bucket_cluster"] = lower_clusters
    scored_df["lower_bucket_label"] = lower_labels
    scored_df["distance_to_lower_bucket"] = lower_distances

    scored_df["upper_distance_ratio_to_assigned"] = upper_distance_ratios
    scored_df["lower_distance_ratio_to_assigned"] = lower_distance_ratios

    scored_df["outlook_flag"] = outlook_flags
    scored_df["outlook_reason"] = outlook_reasons

    return scored_df

## 3. Manual input example

Edit the numbers below to score a private company, competitor, or simulated case.

Currency note: if your model was trained on USD and the input is in EUR, set `FX_TO_MODEL_CURRENCY` to the EUR/USD conversion rate. Ratios are mostly unaffected; `log_assets` is affected.

In [21]:
FX_TO_MODEL_CURRENCY = 1.0  # example: use 1.08 if converting EUR inputs to USD-trained model scale

# manual_input = pd.DataFrame([
#     {
#         "company_name": "Simulated PrivateCo Base Case",
#         "fiscal_year": 2024,
#         "currency": "EUR",
#         "accounting_standard": "IFRS",
#         "major_sector": "Manufacturing / Industrials",
#         "financial_flag": "Non-financial",

#         "assets": 120_000_000,
#         "liabilities": 70_000_000,
#         "equity": 50_000_000,
#         "cash": 8_000_000,
#         "revenue": 80_000_000,
#         "net_income": 3_200_000,
#         "cfo": 9_000_000,
#         "long_term_debt": 38_000_000,
#         "short_term_debt": 7_000_000,
#         "assets_current": 35_000_000,
#         "liabilities_current": 22_000_000,
#         "receivables": 12_000_000,
#         "inventory": 8_000_000,
#         "capex": 4_000_000,
#         "operating_income": 8_000_000,
#         "gross_profit": 21_000_000,
#         "interest_expense": 2_500_000,
#     }
# ])

# manual_input

manual_input_2025 = pd.DataFrame([{
    "company_name": "Manual 2025 Company",
    "fiscal_year": 2025,
    "currency": "USD",
    "major_sector": "Manufacturing / Industrials",
    "financial_flag": "Non-financial",

    # Values converted from thousand BGN to full USD units at 1.7 BGN/USD
    "revenue": 48_585_294.12,
    "assets": 29_721_275.23,
    "current_assets": 10_037_745.82,
    "cash": 34_804.65,
    "receivables": 2_208_235.29,
    "inventory": 7_794_705.88,
    "equity": 9_082_352.94,
    "current_liabilities": 12_722_352.94,
    "liabilities": 20_638_823.53,
    "long_term_debt": 7_910_588.24,
    "short_term_debt": 1_478_235.29,
    "net_income": 949_411.76,
    "cfo": 3_558_235.29,
    "capex": 588_235.29,
    "operating_income": 1_664_705.88,
    "interest_expense": 597_647.06,
}])

manual_input_2025

,company_name,fiscal_year,currency,major_sector,financial_flag,revenue,assets,current_assets,cash,receivables,inventory,equity,current_liabilities,liabilities,long_term_debt,short_term_debt,net_income,cfo,capex,operating_income,interest_expense
0,Manual 2025 Company,2025,USD,Manufacturing / Industrials,Non-financial,48585294.12,29721275.23,10037745.82,34804.65,2208235.29,7794705.88,9082352.94,12722352.94,20638823.53,7910588.24,1478235.29,949411.76,3558235.29,588235.29,1664705.88,597647.06


In [30]:
scored_manual = score_companies(
    manual_input_2025,
    artifact=artifact,
    segment="Non-financial",
    temperature=1.0,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
)

scored_manual_with_outlook = add_adjacent_bucket_distances_and_outlook(
    scored_manual,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

summary_cols_with_outlook = [
    "company_name",
    "fiscal_year",
    "assigned_cluster",
    "cluster_label",
    "risk_rank",
    "cluster_affinity",
    "near_default_affinity",
    "distance_to_assigned_cluster",
    "upper_bucket_cluster",
    "upper_bucket_label",
    "distance_to_upper_bucket",
    "lower_bucket_cluster",
    "lower_bucket_label",
    "distance_to_lower_bucket",
    "outlook_flag",
    "outlook_reason",
    "feature_coverage_pct",
    "warning_flags",
]

missing_cols = [
    col for col in summary_cols_with_outlook
    if col not in scored_manual_with_outlook.columns
]

if missing_cols:
    print("Missing columns:", missing_cols)
    print("Available columns:", list(scored_manual_with_outlook.columns))

existing_summary_cols_with_outlook = [
    col for col in summary_cols_with_outlook
    if col in scored_manual_with_outlook.columns
]

scored_manual_with_outlook[existing_summary_cols_with_outlook]

,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,feature_coverage_pct,warning_flags
0,Manual 2025 Company,2025,1,2 - Moderate risk / lower-investment-grade-like,2,0.36262,0.233,0.523421,4,1 - Low risk / investment-grade-like,0.965737,0,3 - Elevated risk / leveraged,1.212602,Neutral,Company remains materially closer to its assig...,1.0,"current_ratio_below_1, quick_ratio_below_0_5"


In [31]:
ratio_cols = [
    "log_assets",
    "liabilities_to_assets",
    "debt_to_assets",
    "equity_to_assets",
    "cash_to_assets",
    "net_income_to_assets",
    "cfo_to_assets",
    "revenue_to_assets",
    "current_ratio",
    "quick_ratio",
    "interest_coverage",
    "fcf_to_debt",
    "operating_margin",
    "gross_margin",
]

scored_manual[["company_name"] + [c for c in ratio_cols if c in scored_manual.columns]].T

,0
company_name,Manual 2025 Company
log_assets,17.207374
liabilities_to_assets,0.694412
debt_to_assets,0.315896
equity_to_assets,0.305584
cash_to_assets,0.001171
net_income_to_assets,0.031944
cfo_to_assets,0.11972
revenue_to_assets,1.634697
current_ratio,0.788985


## 4. Compare the company to cluster medians

This helps explain why the company was assigned to a specific profile.

In [32]:
def compare_to_cluster_profile(scored_row, artifact):
    """
    Compare one scored company row against the median profile
    of its assigned cluster.

    Expected artifact structure:
    - artifact["cluster_profile_ranked"] contains the cluster profile.
    - scored_row["assigned_cluster"] contains the assigned cluster id.

    Handles cluster profiles with either:
    - raw metric names, e.g. "log_assets"
    - median-prefixed names, e.g. "median_log_assets"
    """

    if "cluster_profile_ranked" not in artifact:
        raise KeyError(
            "artifact['cluster_profile_ranked'] not found. "
            f"Available artifact keys: {list(artifact.keys())}"
        )

    profile = artifact["cluster_profile_ranked"]

    if profile is None or len(profile) == 0:
        raise ValueError("artifact['cluster_profile_ranked'] is empty.")

    if "assigned_cluster" not in scored_row.index:
        raise KeyError("scored_row must contain 'assigned_cluster'.")

    cluster = int(scored_row["assigned_cluster"])

    # Locate assigned cluster profile.
    if "cluster" in profile.columns:
        match = profile.loc[profile["cluster"].astype(int).eq(cluster)]

        if match.empty:
            available_clusters = sorted(
                profile["cluster"].dropna().astype(int).unique().tolist()
            )
            raise KeyError(
                f"Cluster {cluster} not found in cluster_profile_ranked. "
                f"Available clusters: {available_clusters}"
            )

        cluster_profile = match.iloc[0]

    elif cluster in profile.index:
        cluster_profile = profile.loc[cluster]

    else:
        raise KeyError(
            f"Cluster {cluster} not found in cluster_profile_ranked index, "
            "and no 'cluster' column exists."
        )

    base_cols = [
        "log_assets",
        "liabilities_to_assets",
        "debt_to_assets",
        "equity_to_assets",
        "current_ratio",
        "quick_ratio",
        "cash_to_assets",
        "net_income_to_assets",
        "cfo_to_assets",
        "revenue_to_assets",
        "interest_coverage",
        "fcf_to_debt",
        "operating_margin",
        "gross_margin",
        "ebitda_margin",
        "debt_to_ebitda",
        "net_debt_to_ebitda",
        "ebitda_interest_coverage",
    ]

    rows = []

    for col in base_cols:
        if col not in scored_row.index:
            continue

        if col in cluster_profile.index:
            cluster_value = cluster_profile[col]
        elif f"median_{col}" in cluster_profile.index:
            cluster_value = cluster_profile[f"median_{col}"]
        else:
            continue

        company_value = pd.to_numeric(scored_row[col], errors="coerce")
        cluster_value = pd.to_numeric(cluster_value, errors="coerce")

        rows.append({
            "metric": col,
            "company_value": company_value,
            "assigned_cluster_median": cluster_value,
            "difference": company_value - cluster_value,
        })

    if not rows:
        raise ValueError(
            "No comparable metrics found. "
            "Check scored_row columns and artifact['cluster_profile_ranked'] columns."
        )

    comparison = pd.DataFrame(rows).set_index("metric")

    comparison["relative_position"] = np.select(
        [
            comparison["difference"] > 0,
            comparison["difference"] < 0,
        ],
        [
            "above_cluster_median",
            "below_cluster_median",
        ],
        default="equal_to_cluster_median",
    )

    return comparison.round(4)


comparison_to_cluster = compare_to_cluster_profile(
    scored_manual.iloc[0],
    artifact,
)

comparison_to_cluster

,company_value,assigned_cluster_median,difference,relative_position
metric,,,,
log_assets,17.2074,22.9105,-5.7031,below_cluster_median
liabilities_to_assets,0.6944,0.6671,0.0273,above_cluster_median
debt_to_assets,0.3159,0.2948,0.0211,above_cluster_median
equity_to_assets,0.3056,0.3046,0.0010,above_cluster_median
current_ratio,0.7890,0.9986,-0.2097,below_cluster_median
quick_ratio,0.1763,0.3407,-0.1644,below_cluster_median
cash_to_assets,0.0012,0.0287,-0.0275,below_cluster_median
net_income_to_assets,0.0319,0.0336,-0.0016,below_cluster_median
cfo_to_assets,0.1197,0.0808,0.0389,above_cluster_median


## 5. Load input from CSV or Excel

Use this template structure for multiple companies. Required-ish fields are shown below; missing optional fields are allowed, but the data quality score will fall.

In [33]:
input_template = pd.DataFrame([
    {
        "company_name": "ExampleCo A",
        "fiscal_year": 2024,
        "currency": "EUR",
        "accounting_standard": "IFRS",
        "major_sector": "Manufacturing / Industrials",
        "financial_flag": "Non-financial",
        "assets": 120_000_000,
        "liabilities": 70_000_000,
        "equity": 50_000_000,
        "cash": 8_000_000,
        "revenue": 80_000_000,
        "net_income": 3_200_000,
        "cfo": 9_000_000,
        "long_term_debt": 38_000_000,
        "short_term_debt": 7_000_000,
        "assets_current": 35_000_000,
        "liabilities_current": 22_000_000,
        "receivables": 12_000_000,
        "inventory": 8_000_000,
        "capex": 4_000_000,
        "operating_income": 8_000_000,
        "gross_profit": 21_000_000,
        "interest_expense": 2_500_000,
    }
])

template_path = Path("private_company_scoring_template.csv")
input_template.to_csv(template_path, index=False)
print("Template saved to:", template_path)
input_template

Template saved to: private_company_scoring_template.csv


,company_name,fiscal_year,currency,accounting_standard,major_sector,financial_flag,assets,liabilities,equity,cash,revenue,net_income,cfo,long_term_debt,short_term_debt,assets_current,liabilities_current,receivables,inventory,capex,operating_income,gross_profit,interest_expense
0,ExampleCo A,2024,EUR,IFRS,Manufacturing / Industrials,Non-financial,120000000,70000000,50000000,8000000,80000000,3200000,9000000,38000000,7000000,35000000,22000000,12000000,8000000,4000000,8000000,21000000,2500000


In [ ]:
# Option A: load CSV
INPUT_PATH = Path("private_company_scoring_template.csv")
input_companies = pd.read_csv(INPUT_PATH)

# Option B: load Excel instead
# INPUT_PATH = Path("private_company_inputs.xlsx")
# input_companies = pd.read_excel(INPUT_PATH)

input_companies

In [ ]:
scored_file = score_companies(
    input_companies,
    artifact=artifact,
    segment="Non-financial",
    temperature=1.0,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
)

scored_file[summary_cols]

In [ ]:
output_path = Path("private_company_scores.csv")
scored_file.to_csv(output_path, index=False)
print("Scores saved to:", output_path)

## 6. Scenario analysis

This section shows how a company migrates across clusters under stress cases.

In [ ]:
def make_scenarios(base_row):
    base = base_row.copy()
    scenarios = []

    def add_case(name, updates):
        row = base.copy()
        row.update(updates)
        row["scenario"] = name
        scenarios.append(row)

    add_case("base", {})

    add_case("revenue_down_15pct", {
        "revenue": base.get("revenue", np.nan) * 0.85,
        "net_income": base.get("net_income", np.nan) * 0.70,
        "cfo": base.get("cfo", np.nan) * 0.75,
        "operating_income": base.get("operating_income", np.nan) * 0.70,
    })

    add_case("debt_up_25pct", {
        "long_term_debt": base.get("long_term_debt", 0) * 1.25,
        "liabilities": base.get("liabilities", np.nan) + base.get("long_term_debt", 0) * 0.25,
        "cash": max(base.get("cash", 0) - base.get("long_term_debt", 0) * 0.05, 0),
    })

    add_case("cash_burn_case", {
        "cash": max(base.get("cash", 0) * 0.50, 0),
        "cfo": -abs(base.get("cfo", 0)),
        "net_income": -abs(base.get("net_income", 0)),
    })

    add_case("near_default_stress", {
        "liabilities": base.get("assets", np.nan) * 1.10,
        "equity": -base.get("assets", np.nan) * 0.10,
        "long_term_debt": base.get("assets", np.nan) * 0.75,
        "short_term_debt": base.get("assets", np.nan) * 0.15,
        "interest_expense": max(base.get("interest_expense", 0), 1_000_000),
        "operating_income": max(base.get("interest_expense", 1_000_000) * 0.4, 0),
    })

    return pd.DataFrame(scenarios)

scenario_input = make_scenarios(manual_input.iloc[0].to_dict())
scenario_input[["scenario", "assets", "liabilities", "equity", "cash", "revenue", "net_income", "cfo", "long_term_debt", "short_term_debt"]]

In [ ]:
scored_scenarios = score_companies(
    scenario_input,
    artifact=artifact,
    segment="Non-financial",
    temperature=1.0,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
)

scenario_summary_cols = [
    "scenario",
    "assigned_cluster",
    "cluster_label",
    "risk_rank",
    "cluster_affinity",
    "near_default_affinity",
    "liabilities_to_assets",
    "debt_to_assets",
    "equity_to_assets",
    "cfo_to_assets",
    "current_ratio",
    "warning_flags",
]

scored_scenarios[scenario_summary_cols]

## 7. Optional: inspect saved artifact internals

In [ ]:
nonfin_artifact = artifact["segment_artifacts"]["Non-financial"]

print("Features used by model:")
print(nonfin_artifact["feature_cols"])

print("
Cluster labels:")
print(nonfin_artifact.get("cluster_labels"))

print("
Cluster sizes:")
display(nonfin_artifact.get("cluster_sizes"))

print("
Cluster profile:")
display(nonfin_artifact.get("cluster_profile"))